# Lab — Tracking and Structured Pose in an Industrial Workcell

This lab builds one structured visual-state pipeline from transparent primitives:

```text
timestamped frames → noisy detections → association → lifecycle → track identity
                   → keypoints → pose geometry → temporal history → evidence
```

The default path is deterministic, CPU-safe, credential-free, and non-biometric. Ground-truth identities remain evaluation-only. Optional TrackEval, ByteTrack, MMPose/RTMPose, and torchvision Keypoint R-CNN paths are disabled by default.


In [ ]:
from __future__ import annotations

import json
import math
import os
import platform
import statistics
import time
from dataclasses import dataclass, field
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from PIL import Image, ImageDraw, ImageFilter
from scipy.optimize import linear_sum_assignment

SEED = 808
rng = np.random.default_rng(SEED)
ARTIFACT_DIR = Path(".artifacts/tracking_keypoints_pose")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

DEMONSTRATION_THRESHOLD_NOTICE = "Demonstration thresholds for this notebook runtime only."
TRACKER_VERSION = "structured-tracker-teaching-v1"
DATASET_VERSION = "synthetic-workcell-v1"
assert DEMONSTRATION_THRESHOLD_NOTICE.startswith("Demonstration")
print({"python": platform.python_version(), "numpy": np.__version__, "scipy": scipy.__version__})


## Phases 1–2 — Timestamped ground truth and a separate detector stream

The simulator may use hidden identity to render persistent visual appearance. The detector observation contract removes that identity before association. The assertion below guards this boundary.


In [ ]:
FRAME_SIZE = (192, 128)
N_FRAMES = 64
BASE_COLORS = {"container_red": (220, 74, 72), "container_blue": (58, 112, 202), "robot_arm": (238, 174, 61)}
APPEARANCE_BASE = {
    "container_red": np.array([0.94, 0.18, 0.12, 0.23], dtype=float),
    "container_blue": np.array([0.10, 0.25, 0.96, 0.62], dtype=float),
    "robot_arm": np.array([0.70, 0.70, 0.12, 0.40], dtype=float),
}

def make_timestamps(n=N_FRAMES):
    dt = np.full(n, 1 / 20, dtype=float)
    dt[[18, 19, 43]] = [0.10, 0.075, 0.12]  # variable capture and dropped-frame gaps
    return np.cumsum(dt) - dt[0]

def box_from_center(cx, cy, w, h):
    return np.array([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], dtype=float)

def robot_keypoints(frame):
    phase = frame / 10
    base = np.array([42.0, 105.0])
    joint_1 = base + np.array([22 * np.cos(-1.15 + 0.18 * np.sin(phase)), 22 * np.sin(-1.15 + 0.18 * np.sin(phase))])
    joint_2 = joint_1 + np.array([26 * np.cos(-0.70 + 0.45 * np.sin(phase / 1.3)), 26 * np.sin(-0.70 + 0.45 * np.sin(phase / 1.3))])
    end_effector = joint_2 + np.array([18 * np.cos(-0.25 + 0.55 * np.sin(phase / 1.7)), 18 * np.sin(-0.25 + 0.55 * np.sin(phase / 1.7))])
    return np.stack([base, joint_1, joint_2, end_effector])

timestamps = make_timestamps()
ground_truth = []
for frame, timestamp in enumerate(timestamps):
    speed_boost = max(frame - 20, 0) * 0.45
    red_cx = 22 + 1.85 * frame + min(speed_boost, 12)
    blue_cx = 170 - 1.75 * frame - min(speed_boost * 0.8, 10)
    objects = [
        {"ground_truth_id": 1, "class": "container", "style": "container_red", "bbox": box_from_center(red_cx, 73, 23, 18), "keypoints": None, "visibility": None},
        {"ground_truth_id": 2, "class": "container", "style": "container_blue", "bbox": box_from_center(blue_cx, 77, 23, 18), "keypoints": None, "visibility": None},
    ]
    kpts = robot_keypoints(frame)
    robot_box = np.array([kpts[:, 0].min() - 5, kpts[:, 1].min() - 5, kpts[:, 0].max() + 5, kpts[:, 1].max() + 5])
    visibility = np.array([2, 2, 1 if 29 <= frame <= 32 else 2, 1 if 29 <= frame <= 32 else 2])
    objects.append({"ground_truth_id": 3, "class": "robot_arm", "style": "robot_arm", "bbox": robot_box, "keypoints": kpts, "visibility": visibility})
    ground_truth.append({"frame": frame, "timestamp_s": float(timestamp), "camera_id": "camera_a", "image_size": list(FRAME_SIZE), "objects": objects})

def clip_box(box):
    x1, y1, x2, y2 = np.asarray(box, dtype=float)
    return np.array([np.clip(x1, 0, FRAME_SIZE[0] - 2), np.clip(y1, 0, FRAME_SIZE[1] - 2), np.clip(x2, 1, FRAME_SIZE[0] - 1), np.clip(y2, 1, FRAME_SIZE[1] - 1)])

def render_frame(record, camera="camera_a"):
    bg = (237, 241, 246) if camera == "camera_a" else (172, 157, 143)
    image = Image.new("RGB", FRAME_SIZE, bg)
    draw = ImageDraw.Draw(image)
    draw.rectangle((0, 92, FRAME_SIZE[0], FRAME_SIZE[1]), fill=(125, 137, 151))
    for obj in record["objects"]:
        color = BASE_COLORS[obj["style"]]
        box = tuple(np.round(obj["bbox"]).astype(int))
        draw.rounded_rectangle(box, radius=3, fill=color, outline=(27, 47, 67), width=2)
        if obj["keypoints"] is not None:
            pts = [tuple(np.round(point).astype(int)) for point in obj["keypoints"]]
            draw.line(pts, fill=(27, 47, 67), width=3)
            for point in pts:
                draw.ellipse((point[0]-3, point[1]-3, point[0]+3, point[1]+3), fill=(255,255,255), outline=(27,47,67))
    if camera == "camera_c":
        image = image.filter(ImageFilter.GaussianBlur(radius=0.8))
    return image

def simulate_detections(records, camera="camera_a", seed=SEED):
    local = np.random.default_rng(seed + (100 if camera == "camera_c" else 0))
    stream = []
    for record in records:
        detections = []
        frame = record["frame"]
        for obj in record["objects"]:
            crossing = 34 <= frame <= 43 and obj["class"] == "container"
            occluded = 27 <= frame <= 31 and obj["ground_truth_id"] == 1
            miss_p = 0.04 + (0.32 if occluded else 0) + (0.13 if camera == "camera_c" else 0)
            if local.random() < miss_p:
                continue
            jitter_scale = 1.3 + (1.8 if camera == "camera_c" else 0) + (1.0 if crossing else 0)
            jitter = local.normal(0, jitter_scale, 4)
            jitter[2:] += jitter[:2]
            bbox = clip_box(obj["bbox"] + jitter)
            confidence = 0.92 - (0.48 if occluded else 0) - (0.14 if camera == "camera_c" else 0) + local.normal(0, 0.035)
            appearance = APPEARANCE_BASE[obj["style"]] + local.normal(0, 0.035 + (0.08 if camera == "camera_c" else 0), 4)
            appearance = appearance / max(np.linalg.norm(appearance), 1e-9)
            det = {"bbox": bbox, "score": float(np.clip(confidence, 0.05, 0.99)), "class": obj["class"], "appearance": appearance}
            detections.append(det)
        if frame in {14, 41, 55}:
            detections.append({"bbox": box_from_center(130, 25 + frame % 20, 18, 14), "score": 0.58, "class": "container", "appearance": local.normal(size=4)})
        stream.append({"frame": frame, "timestamp_s": record["timestamp_s"], "camera_id": camera, "detections": detections})
    return stream

detections_a = simulate_detections(ground_truth, "camera_a")
assert all("ground_truth_id" not in det for row in detections_a for det in row["detections"])
assert timestamps[19] - timestamps[18] != timestamps[1] - timestamps[0]

sample_frames = [8, 29, 38, 50]
fig, axes = plt.subplots(1, len(sample_frames), figsize=(12, 2.5))
for ax, frame in zip(axes, sample_frames):
    ax.imshow(render_frame(ground_truth[frame]))
    ax.set_title(f"frame {frame} · t={timestamps[frame]:.2f}s")
    ax.axis("off")
fig.tight_layout()
fig.savefig(ARTIFACT_DIR / "ground-truth-frames.png", dpi=150)
plt.close(fig)


## Core primitives — IoU, assignment, and known-answer checks

SciPy solves the rectangular linear-sum assignment. The cost construction, normalization, gates, and lifecycle remain visible course code.


In [ ]:
def box_iou(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    ix1, iy1 = np.maximum(a[:2], b[:2])
    ix2, iy2 = np.minimum(a[2:], b[2:])
    inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    area_a = max(0.0, a[2] - a[0]) * max(0.0, a[3] - a[1])
    area_b = max(0.0, b[2] - b[0]) * max(0.0, b[3] - b[1])
    return inter / max(area_a + area_b - inter, 1e-12)

def center(box):
    box = np.asarray(box, float)
    return (box[:2] + box[2:]) / 2

def hungarian_assignment(cost, max_cost):
    cost = np.asarray(cost, float)
    if cost.size == 0:
        return [], list(range(cost.shape[0])), list(range(cost.shape[1]))
    rows, cols = linear_sum_assignment(cost)
    matches = [(int(i), int(j)) for i, j in zip(rows, cols) if cost[i, j] <= max_cost]
    matched_r, matched_c = {i for i, _ in matches}, {j for _, j in matches}
    return matches, [i for i in range(cost.shape[0]) if i not in matched_r], [j for j in range(cost.shape[1]) if j not in matched_c]

assert box_iou([0, 0, 10, 10], [0, 0, 10, 10]) == 1.0
assert box_iou([0, 0, 10, 10], [20, 20, 30, 30]) == 0.0
known_cost = np.array([[0.1, 0.8], [0.7, 0.2]])
known_matches, _, _ = hungarian_assignment(known_cost, 0.5)
assert known_matches == [(0, 0), (1, 1)]


## Phases 3–8 — `IoUTracker`, lifecycle, motion, appearance, occlusion, and Byte-style recovery

`StructuredTracker` exposes every policy. Hard gates (`class`, `center`, optional minimum IoU, and optional appearance distance) first remove impossible pairs. Hungarian matching then optimizes the weighted cost only across surviving pairs, and `combined_cost_threshold` independently determines whether an assigned pair is accepted. `IoUTracker` is the pure-geometry specialization. The Byte-style path performs high-confidence matching first and a separately configured low-confidence recovery pass second.


In [ ]:
@dataclass
class TrackState:
    track_id: int
    bbox: np.ndarray
    class_name: str
    score: float
    created_at: float
    last_timestamp: float
    hits: int = 1
    age: int = 1
    missed_frames: int = 0
    state: str = "tentative"
    velocity: np.ndarray = field(default_factory=lambda: np.zeros(2, dtype=float))
    appearance: np.ndarray | None = None
    events: list = field(default_factory=list)

class StructuredTracker:
    def __init__(self, iou_gate=None, center_gate=65.0, appearance_gate=None,
                 combined_cost_threshold=0.65, secondary_combined_cost_threshold=0.78,
                 secondary_center_scale=1.35, max_age=3, min_hits=2, use_motion=False,
                 geometry_weight=1.0, appearance_weight=0.0, motion_weight=0.0,
                 high_threshold=0.6, low_threshold=0.2, byte_style=False):
        self.iou_gate = iou_gate
        self.center_gate = center_gate
        self.appearance_gate = appearance_gate
        self.combined_cost_threshold = combined_cost_threshold
        self.secondary_combined_cost_threshold = secondary_combined_cost_threshold
        self.secondary_center_scale = secondary_center_scale
        self.max_age = max_age
        self.min_hits = min_hits
        self.use_motion = use_motion
        self.geometry_weight = geometry_weight
        self.appearance_weight = appearance_weight
        self.motion_weight = motion_weight
        self.high_threshold = high_threshold
        self.low_threshold = low_threshold
        self.byte_style = byte_style
        self.tracks = []
        self.next_id = 1
        self.lifecycle_events = []

    def predict(self, track, timestamp):
        predicted = track.bbox.copy()
        if self.use_motion:
            dt = max(timestamp - track.last_timestamp, 0.0)
            predicted[[0, 2]] += track.velocity[0] * dt
            predicted[[1, 3]] += track.velocity[1] * dt
        return predicted

    def _cost(self, tracks, detections, timestamp, center_scale=1.0):
        matrix = np.full((len(tracks), len(detections)), 1e6, dtype=float)
        for i, track in enumerate(tracks):
            predicted = self.predict(track, timestamp)
            for j, det in enumerate(detections):
                # Hard gates have explicit semantics and run before weighted scoring.
                if det["class"] != track.class_name:
                    continue
                displacement = np.linalg.norm(center(predicted) - center(det["bbox"]))
                if displacement > self.center_gate * center_scale:
                    continue
                iou = box_iou(predicted, det["bbox"])
                if self.iou_gate is not None and iou < self.iou_gate:
                    continue
                appearance = 0.5
                if track.appearance is not None:
                    appearance = 1 - float(np.dot(track.appearance, det["appearance"]) / (np.linalg.norm(track.appearance) * np.linalg.norm(det["appearance"]) + 1e-9))
                if self.appearance_gate is not None and appearance > self.appearance_gate:
                    continue
                geometry = 1 - iou
                motion = min(displacement / max(self.center_gate, 1e-9), 1.0)
                total_weight = self.geometry_weight + self.motion_weight + self.appearance_weight
                matrix[i, j] = (self.geometry_weight * geometry + self.motion_weight * motion + self.appearance_weight * appearance) / max(total_weight, 1e-9)
        return matrix

    def associate(self, tracks, detections, timestamp, secondary=False):
        center_scale = self.secondary_center_scale if secondary else 1.0
        cost = self._cost(tracks, detections, timestamp, center_scale=center_scale)
        max_combined_cost = self.secondary_combined_cost_threshold if secondary else self.combined_cost_threshold
        return (*hungarian_assignment(cost, max_combined_cost), cost)

    def _create(self, det, timestamp):
        state = "confirmed" if self.min_hits <= 1 else "tentative"
        track = TrackState(self.next_id, det["bbox"].copy(), det["class"], det["score"], timestamp, timestamp,
                           state=state, appearance=det["appearance"].copy(), events=[{"event": "birth", "timestamp_s": timestamp}])
        self.lifecycle_events.append({"track_id": self.next_id, "event": "birth", "timestamp_s": timestamp})
        self.next_id += 1
        self.tracks.append(track)

    def _apply_match(self, track, det, timestamp):
        was_lost = track.state == "lost"
        dt = max(timestamp - track.last_timestamp, 1e-6)
        observed_velocity = (center(det["bbox"]) - center(track.bbox)) / dt
        track.velocity = 0.65 * track.velocity + 0.35 * observed_velocity
        track.bbox = det["bbox"].copy()
        track.score = det["score"]
        track.last_timestamp = timestamp
        track.hits += 1
        track.age += 1
        track.missed_frames = 0
        new_appearance = det["appearance"] / max(np.linalg.norm(det["appearance"]), 1e-9)
        track.appearance = 0.8 * track.appearance + 0.2 * new_appearance
        track.appearance /= max(np.linalg.norm(track.appearance), 1e-9)
        if track.hits >= self.min_hits:
            track.state = "confirmed"
        if was_lost:
            track.events.append({"event": "recovered", "timestamp_s": timestamp})
            self.lifecycle_events.append({"track_id": track.track_id, "event": "recovered", "timestamp_s": timestamp})

    def update(self, detections, timestamp, frame):
        candidates = [d for d in detections if d["score"] >= self.low_threshold]
        high = [d for d in candidates if d["score"] >= self.high_threshold]
        low = [d for d in candidates if self.low_threshold <= d["score"] < self.high_threshold]
        matches, unmatched_tracks, unmatched_high, _ = self.associate(self.tracks, high, timestamp)
        for i, j in matches:
            self._apply_match(self.tracks[i], high[j], timestamp)
        remaining_track_indices = unmatched_tracks
        if self.byte_style and low and remaining_track_indices:
            remaining_tracks = [self.tracks[i] for i in remaining_track_indices]
            second, still_unmatched, _, _ = self.associate(remaining_tracks, low, timestamp, secondary=True)
            for local_i, j in second:
                global_i = remaining_track_indices[local_i]
                self._apply_match(self.tracks[global_i], low[j], timestamp)
            remaining_track_indices = [remaining_track_indices[k] for k in still_unmatched]
        for i in remaining_track_indices:
            track = self.tracks[i]
            track.bbox = self.predict(track, timestamp)
            track.last_timestamp = timestamp
            track.age += 1
            track.missed_frames += 1
            if track.state != "lost":
                track.state = "lost"
                track.events.append({"event": "lost", "timestamp_s": timestamp})
                self.lifecycle_events.append({"track_id": track.track_id, "event": "lost", "timestamp_s": timestamp})
        for j in unmatched_high:
            self._create(high[j], timestamp)
        kept = []
        for track in self.tracks:
            if track.missed_frames > self.max_age:
                self.lifecycle_events.append({"track_id": track.track_id, "event": "retired", "timestamp_s": timestamp})
            else:
                kept.append(track)
        self.tracks = kept
        return [{"frame": frame, "timestamp_s": timestamp, "track_id": t.track_id, "bbox": t.bbox.copy(), "class": t.class_name,
                 "score": t.score, "state": t.state, "missed_frames": t.missed_frames} for t in self.tracks if t.state in {"confirmed", "lost"}]

class IoUTracker(StructuredTracker):
    def __init__(self, **kwargs):
        super().__init__(geometry_weight=1.0, appearance_weight=0.0, motion_weight=0.0, use_motion=False, **kwargs)

def run_tracker(stream, **kwargs):
    tracker = StructuredTracker(**kwargs)
    outputs = []
    for row in stream:
        outputs.extend(tracker.update(row["detections"], row["timestamp_s"], row["frame"]))
    return outputs, tracker

# Hand-check the independent hard gates and combined acceptance threshold.
gate_probe = StructuredTracker(iou_gate=0.5, center_gate=50, appearance_gate=0.2, combined_cost_threshold=0.3, min_hits=1)
gate_track = TrackState(1, np.array([0., 0., 10., 10.]), "container", .9, 0., 0., state="confirmed", appearance=np.ones(4) / 2)
far_appearance = np.array([-0.5, -0.5, -0.5, -0.5])
gate_detections = [{"bbox": np.array([1., 1., 11., 11.]), "score": .9, "class": "container", "appearance": far_appearance}]
assert gate_probe._cost([gate_track], gate_detections, 0.05)[0, 0] == 1e6  # appearance hard gate

# Hand-check lifecycle: confirm, become lost, recover, then retire.
probe = IoUTracker(iou_gate=0.1, combined_cost_threshold=0.9, max_age=1, min_hits=2, high_threshold=0.5)
det = {"bbox": np.array([10., 10., 20., 20.]), "score": .9, "class": "container", "appearance": np.ones(4) / 2}
probe.update([det], 0.0, 0); probe.update([det], 0.05, 1)
assert probe.tracks[0].state == "confirmed"
probe.update([], 0.10, 2); assert probe.tracks[0].state == "lost"
probe.update([det], 0.15, 3); assert probe.tracks[0].state == "confirmed"
probe.update([], 0.20, 4); probe.update([], 0.25, 5); assert not probe.tracks


In [ ]:
def evaluate_tracking(gt_records, outputs, evaluation_iou_threshold=0.30):
    output_by_frame = {}
    for row in outputs:
        output_by_frame.setdefault(row["frame"], []).append(row)
    events, total_gt, total_pred = [], 0, 0
    for record in gt_records:
        gt = record["objects"]
        pred = output_by_frame.get(record["frame"], [])
        total_gt += len(gt); total_pred += len(pred)
        cost = np.ones((len(gt), len(pred)), dtype=float)
        for i, g in enumerate(gt):
            for j, p in enumerate(pred):
                if g["class"] == p["class"]:
                    cost[i, j] = 1 - box_iou(g["bbox"], p["bbox"])
        matches, un_gt, un_pred = hungarian_assignment(cost, 1 - evaluation_iou_threshold)
        for i, j in matches:
            events.append({"frame": record["frame"], "timestamp_s": record["timestamp_s"], "event": "match",
                           "ground_truth_id": gt[i]["ground_truth_id"], "track_id": pred[j]["track_id"], "iou": 1 - cost[i, j]})
        events.extend({"frame": record["frame"], "event": "miss", "ground_truth_id": gt[i]["ground_truth_id"], "track_id": None, "iou": 0.0} for i in un_gt)
        events.extend({"frame": record["frame"], "event": "false_track", "ground_truth_id": None, "track_id": pred[j]["track_id"], "iou": 0.0} for j in un_pred)
    table = pd.DataFrame(events)
    matched = table.query("event == 'match'").copy()
    id_switches = []
    fragments = {}
    dominant_counts = 0
    for gt_id in sorted({obj["ground_truth_id"] for row in gt_records for obj in row["objects"]}):
        entity = table[table["ground_truth_id"] == gt_id].sort_values("frame")
        previous = None
        segments = 0
        active = False
        for row in entity.itertuples():
            if row.event == "match":
                if not active:
                    segments += 1
                if previous is not None and row.track_id != previous:
                    id_switches.append({"frame": int(row.frame), "ground_truth_id": int(gt_id), "from_track": int(previous), "to_track": int(row.track_id)})
                previous = row.track_id; active = True
            else:
                active = False
        fragments[int(gt_id)] = max(0, segments - 1)
        counts = matched.loc[matched["ground_truth_id"] == gt_id, "track_id"].value_counts()
        dominant_counts += int(counts.iloc[0]) if len(counts) else 0
    tp = int((table.event == "match").sum()); fn = int((table.event == "miss").sum()); fp = int((table.event == "false_track").sum())
    det_a = tp / max(tp + fp + fn, 1)
    ass_a = dominant_counts / max(tp, 1)
    metrics = {
        "tp": tp, "fp": fp, "fn": fn,
        "precision": tp / max(tp + fp, 1), "recall": tp / max(tp + fn, 1),
        "id_switches": len(id_switches), "fragmentation": int(sum(fragments.values())),
        "mota_teaching": 1 - (fn + fp + len(id_switches)) / max(total_gt, 1),
        "identity_consistency_f1_teaching": 2 * dominant_counts / max(2 * dominant_counts + (tp - dominant_counts) + (total_gt - dominant_counts), 1),
        "detection_accuracy_teaching": det_a, "association_accuracy_teaching": ass_a,
        "hota_like_teaching_not_official": math.sqrt(det_a * ass_a),
    }
    return metrics, table, id_switches, fragments

configs = {
    "geometry_only": dict(iou_gate=.16, combined_cost_threshold=.84, max_age=2, min_hits=1, geometry_weight=1, appearance_weight=0, motion_weight=0, use_motion=False),
    "motion_geometry": dict(iou_gate=None, center_gate=65, combined_cost_threshold=.62, max_age=4, min_hits=1, geometry_weight=.55, appearance_weight=0, motion_weight=.45, use_motion=True),
    "appearance_only": dict(iou_gate=None, center_gate=100, appearance_gate=.50, combined_cost_threshold=.50, max_age=4, min_hits=1, geometry_weight=0, appearance_weight=1, motion_weight=0, use_motion=False),
    "combined": dict(iou_gate=None, center_gate=65, appearance_gate=.50, combined_cost_threshold=.62, max_age=4, min_hits=1, geometry_weight=.45, appearance_weight=.35, motion_weight=.20, use_motion=True),
    "byte_style": dict(iou_gate=None, center_gate=65, appearance_gate=.50, combined_cost_threshold=.62, secondary_combined_cost_threshold=.78, secondary_center_scale=1.35, max_age=4, min_hits=1, geometry_weight=.45, appearance_weight=.35, motion_weight=.20, use_motion=True, byte_style=True),
}
tracking_runs = {}
for name, config in configs.items():
    outputs, tracker = run_tracker(detections_a, **config)
    metrics, events, switches, fragments = evaluate_tracking(ground_truth, outputs)
    tracking_runs[name] = {"outputs": outputs, "tracker": tracker, "metrics": metrics, "events": events, "switches": switches, "fragments": fragments}

tracking_comparison = pd.DataFrame({name: run["metrics"] for name, run in tracking_runs.items()}).T
tracking_comparison.to_csv(ARTIFACT_DIR / "tracking-comparison.csv")
assert set(["geometry_only", "appearance_only", "combined", "byte_style"]).issubset(tracking_comparison.index)
tracking_comparison


In [ ]:
# Phase 4: lifecycle sweep. This pure-IoU experiment couples its cost threshold to IoU intentionally;
# mixed-cost trackers use an independent combined_cost_threshold. Policies are never tuned on final test video.
lifecycle_rows = []
for iou_gate in [0.08, 0.16, 0.28]:
    for max_age in [1, 3, 6]:
        for min_hits in [1, 2]:
            outputs, _ = run_tracker(detections_a, iou_gate=iou_gate, combined_cost_threshold=1-iou_gate,
                                     max_age=max_age, min_hits=min_hits, geometry_weight=1,
                                     appearance_weight=0, motion_weight=0, use_motion=False)
            metric, _, _, _ = evaluate_tracking(ground_truth, outputs)
            lifecycle_rows.append({"iou_gate": iou_gate, "max_age": max_age, "min_hits": min_hits,
                                   "false_tracks": metric["fp"], "fragments": metric["fragmentation"],
                                   "ID_switches": metric["id_switches"], "track_recall": metric["recall"]})
lifecycle_sweep = pd.DataFrame(lifecycle_rows)
lifecycle_sweep.to_csv(ARTIFACT_DIR / "lifecycle-sweep.csv", index=False)
lifecycle_sweep.sort_values(["ID_switches", "fragments", "false_tracks"]).head()


In [ ]:
# Phase 5: scalar recursive-filter intuition on a noisy 1D position trajectory.
# This example has no velocity state and is not the tracker's constant-velocity model.
true_position = 18 + 7.0 * timestamps
measurements = true_position + np.random.default_rng(55).normal(0, 1.8, len(timestamps))

def scalar_recursive_filter_1d(observations, process_variance=0.12, measurement_variance=3.24):
    estimate, covariance = float(observations[0]), 1.0
    filtered = [estimate]
    for measurement in observations[1:]:
        covariance += process_variance
        gain = covariance / (covariance + measurement_variance)
        estimate = estimate + gain * (measurement - estimate)
        covariance = (1 - gain) * covariance
        filtered.append(estimate)
    return np.array(filtered)

filtered = scalar_recursive_filter_1d(measurements)
kalman_intuition_summary = {
    "scope": "scalar position-only recursive filter; no velocity state",
    "raw_rmse": float(np.sqrt(np.mean((measurements - true_position) ** 2))),
    "filtered_rmse": float(np.sqrt(np.mean((filtered - true_position) ** 2))),
}
assert kalman_intuition_summary["filtered_rmse"] < kalman_intuition_summary["raw_rmse"]

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(timestamps, true_position, label="true")
ax.scatter(timestamps, measurements, s=10, alpha=.5, label="measurement")
ax.plot(timestamps, filtered, label="scalar filter intuition")
ax.set(xlabel="timestamp (s)", ylabel="x position", title="Prediction uncertainty + measurement update")
ax.legend(); fig.tight_layout(); fig.savefig(ARTIFACT_DIR / "kalman-intuition.png", dpi=150); plt.close(fig)
kalman_intuition_summary


In [ ]:
# Phase 7: explicit occlusion/crossing slices.
slice_ranges = {"short_occlusion": range(27, 30), "medium_occlusion": range(27, 33), "crossing": range(34, 44)}

def slice_fragment_recoveries(events, frame_range):
    # Count miss-to-match recoveries entirely observed inside this slice.
    subset = events[events.frame.isin(frame_range) & events.ground_truth_id.notna()].copy()
    count = 0
    for _, entity in subset.groupby("ground_truth_id"):
        previous_event = None
        for row in entity.sort_values("frame").itertuples():
            if previous_event == "miss" and row.event == "match":
                count += 1
            previous_event = row.event
    return count

occlusion_rows = []
for model_name, run in tracking_runs.items():
    events = run["events"]
    for slice_name, frame_range in slice_ranges.items():
        subset = events[events.frame.isin(frame_range)]
        matches = int((subset.event == "match").sum())
        misses = int((subset.event == "miss").sum())
        switch_count = sum(event["frame"] in frame_range for event in run["switches"])
        occlusion_rows.append({"tracker": model_name, "slice": slice_name, "recovery_rate": matches / max(matches + misses, 1),
                               "ID_switches": switch_count, "slice_fragment_recoveries": slice_fragment_recoveries(events, frame_range)})
occlusion_slices = pd.DataFrame(occlusion_rows)
assert "fragmentation_total" not in occlusion_slices.columns
occlusion_slices.to_csv(ARTIFACT_DIR / "occlusion-slices.csv", index=False)
occlusion_slices


## Phase 9 — Metric semantics and TrackEval export

The local metrics are teaching diagnostics. They intentionally retain the `_teaching` or `_not_official` suffix. The export below enables an optional official TrackEval run after the learner installs and pins the reference evaluator.


In [ ]:
TRACKEVAL_REPO_REVISION = "12c8791b303e0a0b50f753af204249e622d0281a"  # resolved from the official repository on 2026-09-02
TRACKEVAL_LICENSE = "MIT"

def export_motchallenge(gt_records, outputs, destination):
    destination = Path(destination); destination.mkdir(parents=True, exist_ok=True)
    gt_lines, tracker_lines = [], []
    for record in gt_records:
        for obj in record["objects"]:
            x1, y1, x2, y2 = obj["bbox"]
            gt_lines.append(f'{record["frame"]+1},{obj["ground_truth_id"]},{x1:.3f},{y1:.3f},{x2-x1:.3f},{y2-y1:.3f},1,1,1')
    for row in outputs:
        x1, y1, x2, y2 = row["bbox"]
        tracker_lines.append(f'{row["frame"]+1},{row["track_id"]},{x1:.3f},{y1:.3f},{x2-x1:.3f},{y2-y1:.3f},{row["score"]:.4f},-1,-1,-1')
    (destination / "gt.txt").write_text("\n".join(gt_lines) + "\n", encoding="utf-8")
    (destination / "tracker.txt").write_text("\n".join(tracker_lines) + "\n", encoding="utf-8")
    return {"format": "MOTChallenge-compatible teaching export", "gt_rows": len(gt_lines), "tracker_rows": len(tracker_lines),
            "official_metrics_computed": False, "reference_evaluator": "TrackEval"}

trackeval_export = export_motchallenge(ground_truth, tracking_runs["combined"]["outputs"], ARTIFACT_DIR / "trackeval-input")
assert trackeval_export["official_metrics_computed"] is False


## Phases 10–13 — Heatmaps, visibility, PCK, and OKS-like evaluation

Missing coordinates are represented by `NaN` and evaluated through a separate visibility mask. `oks_like_teaching` is a scale-aware intuition, not the COCO implementation.


In [ ]:
def gaussian_heatmap(coordinate_xy, image_size, heatmap_size, sigma=1.5):
    image_w, image_h = image_size; heat_w, heat_h = heatmap_size
    x = coordinate_xy[0] / image_w * heat_w
    y = coordinate_xy[1] / image_h * heat_h
    yy, xx = np.mgrid[0:heat_h, 0:heat_w]
    return np.exp(-((xx - x) ** 2 + (yy - y) ** 2) / (2 * sigma ** 2)).astype(np.float32)

def decode_heatmap_argmax(heatmap, image_size):
    heat_h, heat_w = heatmap.shape
    y, x = np.unravel_index(np.argmax(heatmap), heatmap.shape)
    return np.array([(x + .5) / heat_w * image_size[0], (y + .5) / heat_h * image_size[1]])

keypoint = robot_keypoints(12)[2]
heatmap = gaussian_heatmap(keypoint, FRAME_SIZE, (32, 32))
decoded = decode_heatmap_argmax(heatmap, FRAME_SIZE)
heatmap_error = float(np.linalg.norm(decoded - keypoint))
assert heatmap.shape == (32, 32) and np.isfinite(heatmap_error)

fig, axes = plt.subplots(1, 2, figsize=(7, 3))
axes[0].imshow(render_frame(ground_truth[12])); axes[0].scatter([keypoint[0]], [keypoint[1]], c="#2F6BFF", s=35); axes[0].axis("off")
axes[1].imshow(heatmap, cmap="magma"); axes[1].set_title(f"decoded error={heatmap_error:.2f}px"); axes[1].axis("off")
fig.tight_layout(); fig.savefig(ARTIFACT_DIR / "keypoint-heatmap.png", dpi=150); plt.close(fig)


In [ ]:
heatmap_resolution_rows = []
for resolution in [16, 32, 64]:
    timings = []
    errors = []
    for _ in range(80):
        started = time.perf_counter()
        hm = gaussian_heatmap(keypoint, FRAME_SIZE, (resolution, resolution))
        recovered = decode_heatmap_argmax(hm, FRAME_SIZE)
        timings.append((time.perf_counter() - started) * 1000)
        errors.append(np.linalg.norm(recovered - keypoint))
    heatmap_resolution_rows.append({"resolution": resolution, "coordinate_error_px": float(np.mean(errors)),
                                    "memory_kib_per_keypoint": hm.nbytes / 1024,
                                    "median_runtime_ms": float(np.median(timings)), "p95_runtime_ms": float(np.percentile(timings, 95))})
heatmap_resolution_comparison = pd.DataFrame(heatmap_resolution_rows)
heatmap_resolution_comparison.to_csv(ARTIFACT_DIR / "heatmap-resolution.csv", index=False)
heatmap_resolution_comparison


In [ ]:
def pck(predicted, target, visibility, scale, alpha=0.10):
    predicted, target, visibility = np.asarray(predicted, float), np.asarray(target, float), np.asarray(visibility, bool)
    valid = visibility & np.isfinite(predicted).all(axis=1) & np.isfinite(target).all(axis=1)
    distances = np.linalg.norm(predicted[valid] - target[valid], axis=1)
    return float(np.mean(distances < alpha * scale)) if valid.any() else float("nan")

def oks_like_teaching(predicted, target, visibility, object_scale, tolerances):
    predicted, target = np.asarray(predicted, float), np.asarray(target, float)
    visibility, tolerances = np.asarray(visibility, bool), np.asarray(tolerances, float)
    valid = visibility & np.isfinite(predicted).all(axis=1) & np.isfinite(target).all(axis=1)
    d2 = np.sum((predicted[valid] - target[valid]) ** 2, axis=1)
    score = np.exp(-d2 / (2 * (object_scale * tolerances[valid]) ** 2 + 1e-12))
    return float(score.mean()) if valid.any() else float("nan")

target = np.array([[10., 10.], [20., 20.], [30., 30.]])
prediction = np.array([[11., 10.], [26., 20.], [np.nan, np.nan]])
visibility = np.array([True, True, False])
assert pck(prediction, target, visibility, scale=50, alpha=.1) == 0.5

same_error_small = oks_like_teaching(target + 4, target, np.ones(3, bool), object_scale=25, tolerances=np.full(3, .2))
same_error_large = oks_like_teaching(target + 4, target, np.ones(3, bool), object_scale=100, tolerances=np.full(3, .2))
assert same_error_small < same_error_large
pck_oks_summary = {"hand_checked_pck": pck(prediction, target, visibility, 50, .1),
                   "same_error_small_object": same_error_small, "same_error_large_object": same_error_large,
                   "metric_label": "OKS-like teaching score; not official COCO OKS"}


## Phases 14–15 — Pose geometry, timestamp velocity, smoothing, jitter, and lag

The robot skeleton is `base → joint_1 → joint_2 → end_effector`. Derived angles inherit point uncertainty. Velocity uses observed timestamps, and smoothing is evaluated against error and lag—not appearance alone.


In [ ]:
def angle_degrees(a, b, c):
    first, second = np.asarray(a) - np.asarray(b), np.asarray(c) - np.asarray(b)
    cosine = np.dot(first, second) / max(np.linalg.norm(first) * np.linalg.norm(second), 1e-12)
    return float(np.degrees(np.arccos(np.clip(cosine, -1, 1))))

def pose_geometry(points):
    points = np.asarray(points, float)
    return {
        "segment_lengths": [float(np.linalg.norm(points[i + 1] - points[i])) for i in range(len(points) - 1)],
        "joint_1_angle_deg": angle_degrees(points[0], points[1], points[2]),
        "joint_2_angle_deg": angle_degrees(points[1], points[2], points[3]),
        "orientation_deg": float(np.degrees(np.arctan2(*(points[-1] - points[0])[::-1]))),
    }

known_right_angle = angle_degrees([0, 0], [1, 0], [1, 1])
assert abs(known_right_angle - 90) < 1e-9
true_pose = robot_keypoints(20)
noisy_pose = true_pose + np.random.default_rng(3).normal(0, 1.4, true_pose.shape)
true_geometry, noisy_geometry = pose_geometry(true_pose), pose_geometry(noisy_pose)
pose_geometry_summary = {"true": true_geometry, "noisy": noisy_geometry,
                         "joint_2_angle_error_deg": abs(true_geometry["joint_2_angle_deg"] - noisy_geometry["joint_2_angle_deg"])}


In [ ]:
true_trajectory = np.stack([robot_keypoints(frame)[-1] for frame in range(N_FRAMES)])
observed_trajectory = true_trajectory + np.random.default_rng(404).normal(0, 1.7, true_trajectory.shape)

def ema(points, alpha):
    result = [np.asarray(points[0], float)]
    for point in points[1:]:
        result.append(alpha * point + (1 - alpha) * result[-1])
    return np.stack(result)

def trajectory_jitter(points):
    velocity = np.diff(points, axis=0)
    acceleration = np.diff(velocity, axis=0)
    return float(np.mean(np.linalg.norm(acceleration, axis=1)))

def temporal_lag_frames(reference, estimate, max_lag=8):
    scores = []
    for lag in range(max_lag + 1):
        ref = reference[: len(reference) - lag or None]
        est = estimate[lag:]
        scores.append(float(np.mean(np.linalg.norm(ref - est, axis=1))))
    return int(np.argmin(scores))

smoothing_rows = []
smoothed_paths = {}
for alpha in [1.0, .65, .35, .15]:
    smoothed = ema(observed_trajectory, alpha)
    smoothed_paths[alpha] = smoothed
    smoothing_rows.append({"alpha": alpha, "mean_keypoint_error_px": float(np.mean(np.linalg.norm(smoothed - true_trajectory, axis=1))),
                           "jitter": trajectory_jitter(smoothed), "lag_frames": temporal_lag_frames(true_trajectory, smoothed)})
smoothing_comparison = pd.DataFrame(smoothing_rows)

timestamp_velocity = np.diff(observed_trajectory, axis=0) / np.diff(timestamps)[:, None]
wrong_frame_velocity = np.diff(observed_trajectory, axis=0)
assert not np.allclose(timestamp_velocity, wrong_frame_velocity)

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(timestamps, true_trajectory[:, 0], label="true")
ax.plot(timestamps, observed_trajectory[:, 0], alpha=.45, label="raw")
ax.plot(timestamps, smoothed_paths[.35][:, 0], label="EMA α=.35")
ax.set(xlabel="timestamp (s)", ylabel="end-effector x", title="Smoothing reduces jitter and adds lag")
ax.legend(); fig.tight_layout(); fig.savefig(ARTIFACT_DIR / "temporal-smoothing.png", dpi=150); plt.close(fig)
smoothing_comparison.to_csv(ARTIFACT_DIR / "smoothing-comparison.csv", index=False)
smoothing_comparison


## Phases 16–17 — Combined track + pose state and failure propagation

Pose observations are assigned to predicted identities through evaluation-time spatial matching. The tracker still never sees hidden identity. The lab reports both a controlled track-ID perturbation, which isolates causality, and a naturally occurring container association failure from the actual tracker run.


In [ ]:
combined_events = tracking_runs["combined"]["events"]
robot_matches = combined_events.query("event == 'match' and ground_truth_id == 3").copy()
track_pose_history = []
pose_rng = np.random.default_rng(99)
for row in robot_matches.itertuples():
    true_points = robot_keypoints(int(row.frame))
    visible = ground_truth[int(row.frame)]["objects"][2]["visibility"] > 0
    predicted = true_points + pose_rng.normal(0, 1.2, true_points.shape)
    predicted[~visible] = np.nan
    geometry = pose_geometry(np.where(np.isfinite(predicted), predicted, true_points))
    track_pose_history.append({"frame": int(row.frame), "timestamp_s": float(timestamps[int(row.frame)]), "track_id": int(row.track_id),
                               "keypoints": predicted, "visibility": visible, "joint_2_angle_deg": geometry["joint_2_angle_deg"]})

def count_pose_identity_discontinuities(history):
    ordered = sorted(history, key=lambda row: row["frame"])
    return sum(ordered[i]["track_id"] != ordered[i-1]["track_id"] for i in range(1, len(ordered)))

normal_discontinuities = count_pose_identity_discontinuities(track_pose_history)
corrupted_history = [dict(row) for row in track_pose_history]
if len(corrupted_history) > 40:
    corrupted_history[38]["track_id"] = corrupted_history[38]["track_id"] + 1000

natural_container_switches = [event for event in tracking_runs["geometry_only"]["switches"] if event["ground_truth_id"] in {1, 2}]
natural_container_association_failure = natural_container_switches[0] if natural_container_switches else None
failure_propagation = {
    "controlled_causal_demonstration": {
        "normal_identity_discontinuities": normal_discontinuities,
        "after_injected_association_error": count_pose_identity_discontinuities(corrupted_history),
        "interpretation": "synthetic track-ID perturbation isolates downstream pose-history ownership",
    },
    "naturally_occurring_tracker_failure": {
        "source": "geometry-only tracker on the crossing-container sequence",
        "identity_switch": natural_container_association_failure,
    },
    "causal_chain": ["detection", "association", "track identity", "keypoint ownership", "pose history"],
}
assert failure_propagation["controlled_causal_demonstration"]["after_injected_association_error"] >= normal_discontinuities
assert natural_container_association_failure is not None
failure_propagation


## Phase 18 — Camera C source shift and stage-level attribution

Camera C changes the rendered image style and the detector-noise simulation. Detection and tracking therefore use the changed observation stream. No pose model runs on those images: pose degradation is a separately injected, stage-specific keypoint-noise proxy. The output table records that boundary explicitly.

```text
Camera C
├── simulated detector degradation from the Camera C observation stream
├── actual propagation through the local tracker
└── injected pose-stage noise proxy (not image-model inference)
```


In [ ]:
detections_c = simulate_detections(ground_truth, "camera_c")
outputs_c, tracker_c = run_tracker(detections_c, **configs["combined"])
metrics_c, events_c, switches_c, fragments_c = evaluate_tracking(ground_truth, outputs_c)

def detector_recall_from_stream(gt_records, stream, threshold=.30):
    matched, total = 0, 0
    for record, row in zip(gt_records, stream):
        total += len(record["objects"])
        cost = np.ones((len(record["objects"]), len(row["detections"])))
        for i, gt in enumerate(record["objects"]):
            for j, det in enumerate(row["detections"]):
                if gt["class"] == det["class"]:
                    cost[i, j] = 1 - box_iou(gt["bbox"], det["bbox"])
        matches, _, _ = hungarian_assignment(cost, 1 - threshold)
        matched += len(matches)
    return matched / max(total, 1)

pose_stage_noise_proxy_px = {"camera_a": 1.2, "camera_c": 2.8}
pose_pck_by_camera = {}
for camera, noise in pose_stage_noise_proxy_px.items():
    local = np.random.default_rng(900 + int(noise * 10))
    scores = []
    for frame in range(N_FRAMES):
        true_points = robot_keypoints(frame)
        pred_points = true_points + local.normal(0, noise, true_points.shape)
        scores.append(pck(pred_points, true_points, np.ones(4, bool), scale=50, alpha=.1))
    pose_pck_by_camera[camera] = float(np.mean(scores))

source_shift = pd.DataFrame([
    {"camera": "camera_a", "detector_recall": detector_recall_from_stream(ground_truth, detections_a),
     "track_recall": tracking_runs["combined"]["metrics"]["recall"], "ID_switches": tracking_runs["combined"]["metrics"]["id_switches"],
     "mota_teaching": tracking_runs["combined"]["metrics"]["mota_teaching"],
     "association_accuracy_teaching": tracking_runs["combined"]["metrics"]["association_accuracy_teaching"], "pose_PCK": pose_pck_by_camera["camera_a"],
     "pose_evidence_source": "injected_pose_stage_noise_proxy", "pose_image_model_inference": False},
    {"camera": "camera_c", "detector_recall": detector_recall_from_stream(ground_truth, detections_c),
     "track_recall": metrics_c["recall"], "ID_switches": metrics_c["id_switches"], "mota_teaching": metrics_c["mota_teaching"],
     "association_accuracy_teaching": metrics_c["association_accuracy_teaching"], "pose_PCK": pose_pck_by_camera["camera_c"],
     "pose_evidence_source": "injected_pose_stage_noise_proxy", "pose_image_model_inference": False},
])
assert not source_shift["pose_image_model_inference"].any()
failure_attribution = pd.DataFrame([
    {"failure": "missed detection", "upstream_source": "detector", "observable_consequence": "track gap"},
    {"failure": "incorrect association", "upstream_source": "tracker", "observable_consequence": "ID switch"},
    {"failure": "bad crop", "upstream_source": "detector/tracker", "observable_consequence": "keypoint error"},
    {"failure": "landmark miss", "upstream_source": "injected pose-stage proxy", "observable_consequence": "pose error"},
    {"failure": "smoothing lag", "upstream_source": "temporal filter", "observable_consequence": "delayed motion"},
    {"failure": "source shift", "upstream_source": "several", "observable_consequence": "mixed degradation"},
])
source_shift_boundary = {
    "detector": "Camera-specific simulated observation stream",
    "tracker": "local tracker propagation over that stream",
    "pose": "injected stage-specific noise proxy; no Camera C image-model inference",
}
source_shift.to_csv(ARTIFACT_DIR / "source-shift.csv", index=False)
failure_attribution.to_csv(ARTIFACT_DIR / "failure-attribution.csv", index=False)
source_shift


## Optional maintained integrations — disabled by default

These guards separate local evidence from downloaded-model observations. Source revisions below are review fields, not instructions to execute remote code blindly. Verify them against the official repositories before enabling.


In [ ]:
ENABLE_OPTIONAL_BYTETRACK = os.getenv("CV_ENABLE_BYTETRACK", "0") == "1"
ENABLE_OPTIONAL_MMPPOSE = os.getenv("CV_ENABLE_MMPPOSE", "0") == "1"
ENABLE_TORCHVISION_KEYPOINT = os.getenv("CV_ENABLE_TORCHVISION_KEYPOINT", "0") == "1"

optional_model_manifests = {
    "trackeval": {"repository": "JonathonLuiten/TrackEval", "revision": TRACKEVAL_REPO_REVISION, "code_license": "MIT",
                  "input": "MOTChallenge text export", "enabled": False},
    "bytetrack": {"repository": "FoundationVision/ByteTrack", "revision": "d1bf0191adff59bc8fcfeaa0b33d3d1642552a99", "code_license": "MIT",
                  "checkpoint_license": "model-specific review required", "enabled": ENABLE_OPTIONAL_BYTETRACK},
    "rtmpose": {"repository": "open-mmlab/mmpose", "revision": "759b39c13fea6ba094afc1fa932f51dc1b11cbf9", "code_license": "Apache-2.0",
                "model": "RTMPose via MMPoseInferencer", "checkpoint_and_data_license": "model-card review required", "enabled": ENABLE_OPTIONAL_MMPPOSE},
    "torchvision_keypointrcnn": {"api": "torchvision.models.detection.keypointrcnn_resnet50_fpn",
                "weights": "KeypointRCNN_ResNet50_FPN_Weights.DEFAULT", "purpose": "human COCO smoke test; not robot landmarks",
                "enabled": ENABLE_TORCHVISION_KEYPOINT},
}

def load_optional_torchvision_keypoint_model():
    if not ENABLE_TORCHVISION_KEYPOINT:
        return None
    from torchvision.models.detection import KeypointRCNN_ResNet50_FPN_Weights, keypointrcnn_resnet50_fpn
    weights = KeypointRCNN_ResNet50_FPN_Weights.DEFAULT
    model = keypointrcnn_resnet50_fpn(weights=weights).eval()
    return model, weights.transforms()

assert load_optional_torchvision_keypoint_model() is None


## Deployment, provenance, and evidence artifact

All thresholds below are demonstration values. A production decision needs target cameras, streams, motion, occlusion, hardware, concurrency, privacy review, recovery behavior, and representative failure costs.


In [ ]:
deployment_contract = {
    "notice": DEMONSTRATION_THRESHOLD_NOTICE,
    "identity_consistency_f1_teaching_min": 0.60,
    "id_switch_rate_max": 0.08,
    "occlusion_recovery_min": 0.65,
    "pose_pck_min": 0.80,
    "p95_pipeline_ms_max": 1000,
    "timestamp_required": True,
    "frame_drop_counter_required": True,
}

def serializable(value):
    if isinstance(value, dict): return {str(k): serializable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)): return [serializable(v) for v in value]
    if isinstance(value, np.ndarray): return value.tolist()
    if isinstance(value, (np.integer,)): return int(value)
    if isinstance(value, (np.floating,)): return float(value)
    if isinstance(value, pd.DataFrame): return value.to_dict(orient="records")
    return value

evidence = {
    "schema_version": "1.0",
    "course": "Beginner 08 — Tracking, Keypoints & Pose",
    "locally_measured_evidence": {
        "dataset_contract": {"version": DATASET_VERSION, "frames": N_FRAMES, "frame_size": FRAME_SIZE, "timestamps_variable": True,
                             "identity_boundary": "ground_truth_id exists only in simulator/evaluator; absent from detector observations"},
        "detector_simulation": {"version": "synthetic-detector-v1", "camera_a_recall": detector_recall_from_stream(ground_truth, detections_a)},
        "tracker_version": TRACKER_VERSION,
        "association_policy": configs,
        "lifecycle_policy_sweep": lifecycle_sweep,
        "tracking_metrics": tracking_comparison,
        "occlusion_slices": occlusion_slices,
        "id_switches": tracking_runs["combined"]["switches"],
        "fragmentation": tracking_runs["combined"]["fragments"],
        "kalman_intuition": kalman_intuition_summary,
        "keypoint_metrics": {"heatmap_error_px": heatmap_error, "resolution": heatmap_resolution_comparison, "pck_oks": pck_oks_summary},
        "pose_geometry": pose_geometry_summary,
        "temporal_jitter_and_lag": smoothing_comparison,
        "failure_propagation": failure_propagation,
        "source_shift": source_shift,
        "source_shift_experimental_boundary": source_shift_boundary,
        "failure_attribution": failure_attribution,
        "trackeval_export": trackeval_export,
        "deployment_contract": deployment_contract,
    },
    "optional_downloaded_model_observations": [],
    "optional_model_manifests": optional_model_manifests,
    "unresolved_production_assumptions": [
        "Synthetic trajectories do not represent production motion, crowding, camera motion, or detector errors.",
        "Teaching identity and HOTA-like metrics are not official MOTChallenge results.",
        "No external tracker or pose checkpoint was downloaded or measured in the default run.",
        "Latency, throughput, frame-drop, and backpressure behavior require target streaming hardware.",
        "Human tracking requires a separate privacy, biometric, legal, consent, retention, and access-control review.",
    ],
    "limitations": ["small procedural corpus", "single-view 2D state", "simple constant-velocity model", "no camera-motion compensation", "no production safety claim"],
}

evidence_path = ARTIFACT_DIR / "course-08-tracking-pose-evidence.json"
evidence_path.write_text(json.dumps(serializable(evidence), indent=2), encoding="utf-8")
assert evidence_path.exists() and not evidence["optional_downloaded_model_observations"]
print(f"Wrote {evidence_path}")


## What you should now be able to explain without code

- Why detector recall and identity continuity must be evaluated separately.
- Why Hungarian assignment cannot rescue a poorly defined cost or gate.
- Why longer track memory can improve recovery and increase wrong re-association.
- Why ByteTrack-style low-score recovery is useful but not risk-free.
- Why a smooth track can carry the wrong identity.
- Why the notebook's HOTA-like decomposition is not official HOTA.
- Why `(0, 0)` is not a safe representation of an absent landmark.
- Why scale and visibility belong in pose evaluation.
- Why small landmark errors can amplify into large joint-angle errors.
- Why EMA smoothing needs an explicit lag measurement.
- Why actual timestamps, not frame numbers, define velocity and backpressure behavior.
- Why a track ID plus time, location, and appearance can become sensitive identity data.

**Next:** Course 09 asks how foundation representations, prompting, open vocabularies, and adapters can support several specialized visual tasks without erasing their evaluation contracts.
